# Nested Incremental-Value Analysis of Epigenetic Risk Score
This notebook implements a formal statistical analysis to determine if the predicted epigenetic risk score carries independent prognostic information beyond chronological age, sex, and standard clinical markers.

We evaluate this on:
1. **FHS Test Split** (Primary held-out cohort)
2. **WHI Cohort** (External validation cohort)

### Nested Model Structure:
* **M0 (Reduced baseline model):** `time, event ~ age + sex + Total_Cholesterol + HDL + Glucose + Triglycerides` (sex dropped for WHI)
* **M1 (Full model):** `M0 + risk_score_std` (where the risk score is standardized to unit SD within the cohort)

### Evaluation Metrics:
1. **Likelihood-Ratio Test (LRT):** Comparing M1 vs M0 to see if adding `risk_score` significantly improves model fit.
2. **Hazard Ratio of risk_score in M1:** Checking the adjusted prognostic effect of the risk score.
3. **Delta C-index:** $C(M1) - C(M0)$ with 1000 resample paired bootstrap 95% confidence intervals.


In [ ]:
import sys
from pathlib import Path

# Resolve repository root and import config
_repo_root = Path.cwd()
while _repo_root != _repo_root.parent:
    if (_repo_root / "config.py").exists():
        if str(_repo_root) not in sys.path:
            sys.path.insert(0, str(_repo_root))
        break
    _repo_root = _repo_root.parent

import config

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from lifelines import CoxPHFitter
from scipy.stats import chi2
from sksurv.metrics import concordance_index_censored

# Configure Styling
sns.set_theme(style="white", context="talk")
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.labelsize": 14,
    "axes.titlesize": 16,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "figure.titlesize": 18
})

# Setup directories
BASE_DIR = config.PROJECT_ROOT
DATA_DIR = config.DATA_DIR
PRED_DIR = config.PREDICTIONS_DIR
PLOT_DIR = BASE_DIR / "plots" / "combined"

PLOT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
np.random.seed(SEED)

print("Environment set up successfully. Base directory:", BASE_DIR)

In [ ]:
# Load raw datasets
fhs_data = pd.read_csv(DATA_DIR / "FHS_non_normalized.csv", low_memory=False)
fhs_data = fhs_data.rename(columns={"Share_ID": "patient_id"})

whi_data = pd.read_csv(DATA_DIR / "WHI_non_normalized.csv", low_memory=False)
whi_data = whi_data.rename(columns={"SUBJECT_ID": "patient_id"})

# Identify features
DROP_COLS = ["patient_id", "SUBJECT_ID", "Share_ID", "dbGaP_Subject_ID", "time", "event", "cohort", "batch", "age", "sex"]
fhs_features = [c for c in fhs_data.columns if c not in DROP_COLS]
whi_features = [c for c in whi_data.columns if c not in DROP_COLS]
common_features = sorted(list(set(fhs_features).intersection(set(whi_features))))

FHS = fhs_data[["patient_id", "time", "event", "age", "sex"] + common_features].copy()
WHI = whi_data[["patient_id", "time", "event", "age", "sex"] + common_features].copy()

# Stratified Splitting for FHS (Train 60%, Val 20%, Test 20%)
FHS["event_stratify"] = FHS["event"].astype(int)

train_val_df, test_df = train_test_split(
    FHS, test_size=0.20, random_state=SEED, stratify=FHS["event_stratify"]
)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.25, random_state=SEED, stratify=train_val_df["event_stratify"]
)

train_df = train_df.drop(columns=["event_stratify"]).reset_index(drop=True)
val_df = val_df.drop(columns=["event_stratify"]).reset_index(drop=True)
test_df = test_df.drop(columns=["event_stratify"]).reset_index(drop=True)
FHS = FHS.drop(columns=["event_stratify"])

In [ ]:
# Load existing CpG+clinical predictions (V3_OPTUNA_SS model)
train_preds = pd.read_csv(PRED_DIR / "coxnet_train_V3_OPTUNA_SS.csv")
val_preds = pd.read_csv(PRED_DIR / "coxnet_val_V3_OPTUNA_SS.csv")
test_preds = pd.read_csv(PRED_DIR / "coxnet_test_V3_OPTUNA_SS.csv")
whi_preds = pd.read_csv(PRED_DIR / "coxnet_whi_V3_OPTUNA_SS.csv")

def merge_predictions(df, preds):
    df = df.copy()
    df['patient_id'] = df['patient_id'].astype(int)
    preds = preds.copy()
    preds['patient_id'] = preds['patient_id'].astype(int)
    m = pd.merge(df, preds[['patient_id', 'risk_score']], on='patient_id')
    assert len(m) == len(df), f"Mismatch in length: {len(df)} vs {len(m)}"
    return m
    
train_df = merge_predictions(train_df, train_preds)
val_df = merge_predictions(val_df, val_preds)
test_df = merge_predictions(test_df, test_preds)
WHI_df = merge_predictions(WHI, whi_preds)

In [ ]:
# Standardize risk_score to unit SD within each cohort
test_df_clean = test_df.dropna().copy()
test_df_clean['risk_score_std'] = (test_df_clean['risk_score'] - test_df_clean['risk_score'].mean()) / test_df_clean['risk_score'].std()

WHI_df_clean = WHI_df.dropna().copy()
WHI_df_clean['risk_score_std'] = (WHI_df_clean['risk_score'] - WHI_df_clean['risk_score'].mean()) / WHI_df_clean['risk_score'].std()

clinical_markers = ["Total_Cholesterol", "HDL", "Glucose", "Triglycerides"]

cohorts = {
    'Test': (test_df_clean, True),  # True: include sex as covariate
    'WHI': (WHI_df_clean, False)    # False: WHI is all-female, sex has zero variance, drop it
}

In [ ]:
results = {}

for name, (df, include_sex) in cohorts.items():
    covs_M0 = ["age"] + clinical_markers
    if include_sex and df["sex"].nunique() > 1:
        covs_M0 = ["age", "sex"] + clinical_markers
        
    covs_M1 = covs_M0 + ["risk_score_std"]
    
    # Fit M0
    cph0 = CoxPHFitter()
    cph0.fit(df[covs_M0 + ['time', 'event']], duration_col='time', event_col='event')
    
    # Fit M1
    cph1 = CoxPHFitter()
    cph1.fit(df[covs_M1 + ['time', 'event']], duration_col='time', event_col='event')
    
    # LRT
    LR_stat = 2.0 * (cph1.log_likelihood_ - cph0.log_likelihood_)
    p_lrt = 1.0 - chi2.cdf(LR_stat, df=1)
    
    # M1 summary and risk score HR
    summary1 = cph1.summary
    hr = summary1.loc['risk_score_std', 'exp(coef)']
    hr_lower = summary1.loc['risk_score_std', 'exp(coef) lower 95%']
    hr_upper = summary1.loc['risk_score_std', 'exp(coef) upper 95%']
    p_hr = summary1.loc['risk_score_std', 'p']
    
    # Delta C-index on original sample
    pred_M0 = cph0.predict_partial_hazard(df[covs_M0])
    pred_M1 = cph1.predict_partial_hazard(df[covs_M1])
    
    event_arr = df['event'].values.astype(bool)
    time_arr = df['time'].values
    
    c0 = concordance_index_censored(event_arr, time_arr, pred_M0)[0]
    c1 = concordance_index_censored(event_arr, time_arr, pred_M1)[0]
    delta_c = c1 - c0
    
    results[name] = {
        'model0': cph0,
        'model1': cph1,
        'covs0': covs_M0,
        'covs1': covs_M1,
        'LR_stat': LR_stat,
        'p_lrt': p_lrt,
        'hr': hr,
        'hr_lower': hr_lower,
        'hr_upper': hr_upper,
        'p_hr': p_hr,
        'c0': c0,
        'c1': c1,
        'delta_c': delta_c,
        'df': df
    }
    
    print(f"[{name}] LRT p-value: {p_lrt:.4f}, Delta C-index: {delta_c:.4f}")

In [ ]:
n_boot = 1000

for name in ['Test', 'WHI']:
    res = results[name]
    df = res['df']
    covs0 = res['covs0']
    covs1 = res['covs1']
    
    rng = np.random.RandomState(SEED)
    boot_indices = rng.choice(len(df), size=(n_boot, len(df)), replace=True)
    
    delta_boots = []
    for b in range(n_boot):
        idx = boot_indices[b]
        boot_df = df.iloc[idx].reset_index(drop=True)
        
        try:
            cph0_b = CoxPHFitter()
            cph0_b.fit(boot_df[covs0 + ['time', 'event']], duration_col='time', event_col='event')
            cph1_b = CoxPHFitter()
            cph1_b.fit(boot_df[covs1 + ['time', 'event']], duration_col='time', event_col='event')
            
            pred_M0_b = cph0_b.predict_partial_hazard(boot_df[covs0])
            pred_M1_b = cph1_b.predict_partial_hazard(boot_df[covs1])
            
            ev_b = boot_df['event'].values.astype(bool)
            ti_b = boot_df['time'].values
            
            ci0 = concordance_index_censored(ev_b, ti_b, pred_M0_b)[0]
            ci1 = concordance_index_censored(ev_b, ti_b, pred_M1_b)[0]
            delta_boots.append(ci1 - ci0)
        except Exception:
            pass
            
    delta_boots = np.array(delta_boots)
    ci_lower = np.percentile(delta_boots, 2.5)
    ci_upper = np.percentile(delta_boots, 97.5)
    
    results[name]['ci_lower'] = ci_lower
    results[name]['ci_upper'] = ci_upper
    results[name]['boot_runs'] = delta_boots
    
    print(f"[{name}] Bootstrap 95% CI: ({ci_lower:.4f} to {ci_upper:.4f})")

In [ ]:
# Plot 1: Forest Plot of adjusted Hazard Ratios for FHS Test split M1
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharex=True)

for idx, name in enumerate(['Test', 'WHI']):
    ax = axes[idx]
    res = results[name]
    summary = res['model1'].summary
    
    covariates = summary.index.tolist()
    hrs = summary['exp(coef)'].values
    lowers = summary['exp(coef) lower 95%'].values
    uppers = summary['exp(coef) upper 95%'].values
    p_values = summary['p'].values
    
    y_positions = np.arange(len(covariates))
    
    # Colors: highlight risk_score_std in purple
    colors = ['#3F51B5' if cov != 'risk_score_std' else '#9C27B0' for cov in covariates]
    
    for i, (cov, hr, l, u, p) in enumerate(zip(covariates, hrs, lowers, uppers, p_values)):
        ax.errorbar(
            x=hr,
            y=i,
            xerr=[[hr - l], [u - hr]],
            fmt='o',
            color=colors[i],
            ecolor='#37474F',
            elinewidth=2,
            capsize=4,
            markersize=8
        )
        p_str = f"p = {p:.4f}" if p >= 0.0001 else f"p = {p:.2e}"
        ax.text(u * 1.05, i, f"HR={hr:.2f} ({p_str})", va='center', fontsize=10, color='#37474F', fontweight='semibold')
        
    ax.set_yticks(y_positions)
    ax.set_yticklabels(covariates, fontweight='bold')
    ax.axvline(x=1.0, color='gray', linestyle='--', linewidth=1.5)
    ax.set_xlabel("Hazard Ratio (95% CI)")
    ax.set_title(f"Adjusted Hazard Ratios (Model M1) on {name}", fontweight='bold')
    ax.grid(axis='x', linestyle='--', alpha=0.5)

plt.tight_layout()
plot_path = PLOT_DIR / "nested_analysis_forest.png"
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
print(f"Saved forest plot to: {plot_path}")
plt.show()

In [ ]:
# Plot 2: Paired Bootstrap Differences DeltaC Distribution (Test and WHI)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, split_name in enumerate(['Test', 'WHI']):
    ax = axes[idx]
    res = results[split_name]
    boots = res['boot_runs']
    obs = res['delta_c']
    
    sns.histplot(boots, kde=True, color='purple', ax=ax, edgecolor='white', alpha=0.6)
    ax.axvline(x=0, color='gray', linestyle='--', linewidth=1.5, label='Null (DeltaC = 0)')
    ax.axvline(x=obs, color='crimson', linestyle='-', linewidth=2.0, label=f'Observed DeltaC = {obs:.4f}')
    
    p_str = f"LRT p = {res['p_lrt']:.4f}" if res['p_lrt'] >= 0.0001 else f"LRT p = {res['p_lrt']:.2e}"
    ax.set_title(f"Bootstrap Distribution of DeltaC on {split_name}\n(95% CI: {res['ci_lower']:.4f} to {res['ci_upper']:.4f}, {p_str})", fontsize=13)
    ax.set_xlabel("Delta C-index (M1 - M0)")
    ax.set_ylabel("Count")
    ax.legend(fontsize=10, loc='upper right')

plt.suptitle("Paired Nested Delta C-index Bootstrap Distributions", fontweight='bold', y=1.02)
plt.tight_layout()
plot_path = PLOT_DIR / "nested_analysis_deltas.png"
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
print(f"Saved delta distribution plot to: {plot_path}")
plt.show()

In [ ]:
# Print formal statistics ready to paste
print("\n" + "="*50)
print("FORMAL STATISTICS OUTPUT READY TO PASTE")
print("="*50)

for cohort_name in ['Test', 'WHI']:
    res = results[cohort_name]
    p_lrt_val = res['p_lrt']
    p_lrt_str = f"{p_lrt_val:.4e}" if p_lrt_val < 0.0001 else f"{p_lrt_val:.4f}"
    p_hr_val = res['p_hr']
    p_hr_str = f"{p_hr_val:.4e}" if p_hr_val < 0.0001 else f"{p_hr_val:.4f}"
    
    print(f"\n[{cohort_name} Cohort]")
    print(f'LRT p = {p_lrt_str}; DeltaC-index = {res["delta_c"]:.4f} (95% CI {res["ci_lower"]:.4f} to {res["ci_upper"]:.4f})')
    print(f'M1 risk_score HR = {res["hr"]:.4f} (95% CI {res["hr_lower"]:.4f} to {res["hr_upper"]:.4f}, p = {p_hr_str})')